# 0 Preprocesing

## 0.1 Daten laden

In [1]:
import pandas as pd

In [3]:
# df_full = pd.read_excel('ground_truth_filtered.xlsx')
df_LLM = pd.read_excel('titelannotationen_gesamt_normalisiert.xlsx')

In [4]:
df_LLM.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7776 entries, 0 to 7775
Data columns (total 18 columns):
 #   Column                                         Non-Null Count  Dtype 
---  ------                                         --------------  ----- 
 0   row_id                                         7776 non-null   int64 
 1   article_id                                     7776 non-null   object
 2   Headline_GER                                   7776 non-null   object
 3   Language                                       7776 non-null   object
 4   Economic                                       7776 non-null   int64 
 5   Capacity and resources                         7776 non-null   int64 
 6   Morality                                       7776 non-null   int64 
 7   Fairness and equality                          7776 non-null   int64 
 8   Legality, constitutionality and jurisprudence  7776 non-null   int64 
 9   Policy prescription and evaluation             7776 non-null   

## 0.2 Neue Spalte: 'other'

In [5]:
cols = df_LLM.columns[4:]

df_LLM15 = df_LLM.copy()

df_LLM15['other'] = (
    df_LLM15[cols]
    .eq(0)
    .all(axis=1)
    .astype(int)
)


In [6]:
df_LLM15['other'].value_counts()

,count
other,
1,5284
0,2492


In [7]:
df_LLM15.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7776 entries, 0 to 7775
Data columns (total 19 columns):
 #   Column                                         Non-Null Count  Dtype 
---  ------                                         --------------  ----- 
 0   row_id                                         7776 non-null   int64 
 1   article_id                                     7776 non-null   object
 2   Headline_GER                                   7776 non-null   object
 3   Language                                       7776 non-null   object
 4   Economic                                       7776 non-null   int64 
 5   Capacity and resources                         7776 non-null   int64 
 6   Morality                                       7776 non-null   int64 
 7   Fairness and equality                          7776 non-null   int64 
 8   Legality, constitutionality and jurisprudence  7776 non-null   int64 
 9   Policy prescription and evaluation             7776 non-null   

## 0.3 LLM Daten filtern
Generally keep all rows when 'other' == 0. When 'other' == 1, only keep rows, where 'Language' is 'chn' or 'ger'

In [8]:
df_LLM15 = df_LLM15[
    (df_LLM15['other'] == 0) |
    df_LLM15['Language'].isin(['ge', 'chn'])
]

In [9]:
# check if filtering worked:
df_LLM15['other'].value_counts()

,count
other,
0,2492
1,98


In [10]:
df_LLM15.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2590 entries, 0 to 7773
Data columns (total 19 columns):
 #   Column                                         Non-Null Count  Dtype 
---  ------                                         --------------  ----- 
 0   row_id                                         2590 non-null   int64 
 1   article_id                                     2590 non-null   object
 2   Headline_GER                                   2590 non-null   object
 3   Language                                       2590 non-null   object
 4   Economic                                       2590 non-null   int64 
 5   Capacity and resources                         2590 non-null   int64 
 6   Morality                                       2590 non-null   int64 
 7   Fairness and equality                          2590 non-null   int64 
 8   Legality, constitutionality and jurisprudence  2590 non-null   int64 
 9   Policy prescription and evaluation             2590 non-null   int64

In [20]:
dist_overview = (
df_LLM15.iloc[:, 4:]
.sum()
.div(df_LLM15.iloc[:, 4:].sum().sum())
.mul(100)
.round(2)
.sort_values()
)

dist_overview

,0
Quality of life,1.00
Cultural identity,1.29
Public opinion,1.39
Fairness and equality,1.71
other,3.16
Capacity and resources,4.19
Morality,4.54
Health and safety,5.12
Crime and punishment,5.54
Economic,6.99


## 0.4 Sprachspezifische Daten erstellen

In [ ]:
df_LLM15['Language'].unique()

array(['chn', 'english_mfc', 'en', 'fr', 'ge', 'it', 'po'], dtype=object)

In [ ]:
df_LLM15.columns

Index(['row_id', 'article_id', 'Headline_GER', 'Language', 'Economic',
       'Capacity and resources', 'Morality', 'Fairness and equality',
       'Legality, constitutionality and jurisprudence',
       'Policy prescription and evaluation', 'Crime and punishment',
       'Security and defense', 'Health and safety', 'Quality of life',
       'Cultural identity', 'Public opinion', 'Political',
       'External regulation and reputation', 'other'],
      dtype='object')

In [ ]:
# ---- NORMAL TRAINING LOOPS
# GER
# df_full_ger = df_full[df_full['Language']=='ge'].drop(columns='Language')
df_LLM_ger = df_LLM15[df_LLM15['Language']=='ge'].drop(columns=['row_id', 'article_id', 'Language'])

# EUR minus GER
# df_full_eur = df_full[df_full['Language']!='ge'].drop(columns='Language')
df_LLM_eur_minus = df_LLM15[~df_LLM15['Language'].isin(['ge', 'chn', 'english_mfc'])].drop(columns=['row_id', 'article_id', 'Language'])

# EUR plus GER
# df_full_eur = df_full[df_full['Language']!='ge'].drop(columns='Language')
df_LLM_eur_plus = df_LLM15[~df_LLM15['Language'].isin(['chn', 'english_mfc'])].drop(columns=['row_id', 'article_id', 'Language'])

# all EUR and MFC
# df_full_eur = df_full[df_full['Language']!='ge'].drop(columns='Language')
df_LLM_eur_mfc = df_LLM15[~df_LLM15['Language'].isin(['chn'])].drop(columns=['row_id', 'article_id', 'Language'])

# all EUR and CHN
# df_full_eur = df_full[df_full['Language']!='ge'].drop(columns='Language')
df_LLM_eur_chn = df_LLM15[~df_LLM15['Language'].isin(['english_mfc'])].drop(columns=['row_id', 'article_id', 'Language'])

# complete dataset
# df_full_eur = df_full[df_full['Language']!='ge'].drop(columns='Language')
df_LLM_comp = df_LLM15.drop(columns=['row_id', 'article_id', 'Language'])

# ---- TRANSFER TRAINING LOOPS
# all minus GER & MFC & CHN
df_LLM_ex_GerChnMFC = df_LLM15[~df_LLM15['Language'].isin(['ge', 'english_mfc', 'chn'])].drop(columns=['row_id', 'article_id', 'Language'])

# all minus GER & MFC
df_LLM_ex_GerMFC = df_LLM15[~df_LLM15['Language'].isin(['ge', 'english_mfc'])].drop(columns=['row_id', 'article_id', 'Language'])

# all minus GER & CHN
df_LLM_ex_GerChn = df_LLM15[~df_LLM15['Language'].isin(['ge', 'chn'])].drop(columns=['row_id', 'article_id', 'Language'])

# all minus GER
df_LLM_ex_Ger = df_LLM15[~df_LLM15['Language'].isin(['ge'])].drop(columns=['row_id', 'article_id', 'Language'])

In [ ]:
print('---- Normal Training Data')
print(f'GER: {len(df_LLM_ger)}')
print(f'EUR minus: {len(df_LLM_eur_minus)}')
print(f'EUR plus: {len(df_LLM_eur_plus)}')
print(f'EUR & MFC: {len(df_LLM_eur_mfc)}')
print(f'EUR & CHN: {len(df_LLM_eur_chn)}')
print(f'complete: {len(df_LLM_comp)}')
print('---- Transfer Training Data')
print(f'All minus GER & CHN & MFC: {len(df_LLM_ex_GerChnMFC)}')
print(f'All minus GER & MFC: {len(df_LLM_ex_GerMFC)}')
print(f'All minus GER & CHN: {len(df_LLM_ex_GerChn)}')
print(f'All minus GER: {len(df_LLM_ex_Ger)}')

---- Normal Training Data
GER: 177
EUR minus: 819
EUR plus: 996
EUR & MFC: 2279
EUR & CHN: 1307
complete: 2590
---- Transfer Training Data
All minus GER & CHN & MFC: 819
All minus GER & MFC: 1130
All minus GER & CHN: 2102
All minus GER: 2413


# 1 Fine Tuning

In [ ]:
!pip install iterative-stratification

In [ ]:
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import EvalPrediction
from datasets import Dataset, DatasetDict

# 1. Python RNG
random.seed(42)

# 2. NumPy RNG
np.random.seed(42)

# 3. PyTorch RNG (CPU)
torch.manual_seed(42)

# 4. PyTorch RNG (GPU)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)

# 5. cuDNN determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 6. HF Transformers (Trainer)
from transformers import set_seed
set_seed(42)

## 1.1 Modell-Spezifikationen

### 1.1.1 Stabile Hyperparamter

In [ ]:
model_id = "LSX-UniWue/ModernGBERT_134M"

tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    # Truncate and pad to a standard max length for headlines
    return tokenizer(examples[text_col], padding="max_length", truncation=True, max_length=128)

config = AutoConfig.from_pretrained(model_id)

config.hidden_dropout_prob = 0.10          # Encoder hidden dropout
config.attention_probs_dropout_prob = 0.10 # Attention dropout
config.classifier_dropout_prob = 0.20      # Classification-head dropout

config.problem_type = "multi_label_classification"

config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/729k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

### 1.1.2 Funktion für Berechnung der Metriken

In [ ]:
def compute_metrics(p: EvalPrediction):
    predictions = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    labels = p.label_ids

    # Apply sigmoid since it's a multi-label setup
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))

    # Apply a 0.5 threshold to convert probabilities to binary predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= 0.5)] = 1

    # Overall aggregate metrics
    f1_micro = f1_score(y_true=labels, y_pred=y_pred, average='micro')
    f1_macro = f1_score(y_true=labels, y_pred=y_pred, average='macro', zero_division=0)
    precision = precision_score(y_true=labels, y_pred=y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true=labels, y_pred=y_pred, average='macro', zero_division=0)
    accuracy = accuracy_score(y_true=labels, y_pred=y_pred)

    metrics = {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall
    }

    # Per-label metrics (returns array of shape [num_labels])
    f1_per_label = f1_score(y_true=labels, y_pred=y_pred, average=None, zero_division=0)

    label_names = [f"label_{i}" for i in range(labels.shape[1])]

    for name, f1 in zip(label_names, f1_per_label):
        metrics[f'f1_{name}'] = float(f1)

    return metrics

### 1.1.3 Trainer Setup

In [ ]:
general_training_args = {
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "learning_rate": 1e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,
    "num_train_epochs": 10,
    "weight_decay": 0.05,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "logging_steps": 50,

    # Disk space optimization
    "save_total_limit": 2,
    "save_only_model": True,
}

## 1.2 German: Volltext- vs. Titelannotation

### 1.2.1 Volltext-Annotation

In [ ]:
# df_full_ger = df_full[df_full['Language']=='ge'].drop(columns='Language')

In [ ]:
# df_full_ger.head(3)

In [ ]:
# # Define the text column and extract the label columns
# text_col = 'Headline_GER'
# label_cols = [col for col in df_full_ger.columns if col != text_col]

In [ ]:
# Y = df_full_ger[label_cols].values

# # FIRST SPLIT: 80% Train, 20% Rest
# msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
# train_idx, rest_idx = next(msss1.split(df_full_ger, Y))

# train_df = df_full_ger.iloc[train_idx].copy()
# rest_df = df_full_ger.iloc[rest_idx].copy()
# Y_rest = Y[rest_idx]

# # SECOND SPLIT: Split Rest into 12% Dev / 8% Test
# msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
# dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

# dev_df = rest_df.iloc[dev_sub_idx].copy()
# test_df = rest_df.iloc[test_sub_idx].copy()

# # View
# print(f"Train size: {len(train_df)}")
# print(f"Dev size:   {len(dev_df)}")
# print(f"Test size:  {len(test_df)}")

In [ ]:
# # Convert individual binary columns into a single float list per row
# for split_df in [train_df, dev_df, test_df]:
#     split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# # Create  DatasetDict
# dataset = DatasetDict({
#     'train': Dataset.from_pandas(train_df, preserve_index=False),
#     'dev': Dataset.from_pandas(dev_df, preserve_index=False),
#     'test': Dataset.from_pandas(test_df, preserve_index=False)
# })

In [ ]:
# # Varying Hyerparameters
# tokenized_datasets = dataset.map(tokenize_function, batched=True)
# num_labels = len(label_cols)
# config.num_labels = num_labels

In [ ]:
# # Load the model with the modified configuration
# def new_model_init():
#     print("Initializing new model")
#     return AutoModelForSequenceClassification.from_pretrained(
#         model_id,
#         config=config
#     )

# trainer = Trainer(
#     model_init=new_model_init,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["dev"],
#     compute_metrics=compute_metrics
# )

In [ ]:
# # Execute the training loop
# trainer.train()

In [ ]:
# Reset Accelerator state to fix the AttributeError
# from accelerate.state import AcceleratorState
# AcceleratorState._reset_state()

In [ ]:
# # Final evaluation against the unseen test set
# print("\n--- Final Test Set Evaluation ---")
# test_results = trainer.evaluate(tokenized_datasets["test"])
# print(test_results)

### 1.2.2 Titel-Annotation (LLM)

In [ ]:
# df_LLM_ger = df_LLM[df_LLM['Language']=='ge'].drop(columns=[
#     'Language',
#     # Labels
#     'Quality of life',
#     'Capacity and resources',
#     'Fairness and equality',
#     'Public opinion',
#     'Cultural identity',
#     'Policy prescription and evaluation'])

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_LLM-GER",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
df_LLM_ger.head()

,Headline_GER,Economic,Capacity and resources,Morality,Fairness and equality,"Legality, constitutionality and jurisprudence",Policy prescription and evaluation,Crime and punishment,Security and defense,Health and safety,Quality of life,Cultural identity,Public opinion,Political,External regulation and reputation,other
7106,Arbeitskampf gegen Amazon: Die Unsichtbaren au...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
7107,"""CO₂-Kompensieren löst das Problem nicht""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
7108,Comeback für Klimakiller,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
7109,Pokern ohne As im Ärmel,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
7110,Teil staatlicher Politik,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_ger.columns if col != text_col]

In [ ]:
Y = df_LLM_ger[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, rest_idx = next(msss1.split(df_LLM_ger, Y))

train_df = df_LLM_ger.iloc[train_idx].copy()
rest_df = df_LLM_ger.iloc[rest_idx].copy()
Y_rest = Y[rest_idx]

# SECOND SPLIT: Split Rest into 12% Dev / 8% Test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

dev_df = rest_df.iloc[dev_sub_idx].copy()
test_df = rest_df.iloc[test_sub_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 141
Dev size:   22
Test size:  14


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/141 [00:00<?, ? examples/s]

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
def new_model_init():
    print("Initializing new model")
    return AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    )

trainer = Trainer(
    model_init=new_model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,No log,0.417313,0.318841,0.155556,0.090909,0.133333,0.200000,0.666667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.666667
2,No log,0.252190,0.242424,0.031373,0.181818,0.033333,0.029630,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.470588
3,No log,0.223866,0.200000,0.028571,0.136364,0.040000,0.022222,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.428571
4,No log,0.216942,0.303030,0.039216,0.227273,0.041667,0.037037,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.588235
5,No log,0.214912,0.294118,0.037037,0.227273,0.037037,0.037037,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.555556
6,0.264300,0.213915,0.303030,0.100000,0.227273,0.104762,0.096296,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000
7,0.264300,0.213772,0.285714,0.098039,0.227273,0.100000,0.096296,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.470588
8,0.264300,0.213388,0.285714,0.098039,0.227273,0.100000,0.096296,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.470588
9,0.264300,0.213308,0.235294,0.091667,0.181818,0.095238,0.088889,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.375000
10,0.264300,0.213177,0.235294,0.091667,0.181818,0.095238,0.088889,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.375000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=90, training_loss=0.18408127890692816, metrics={'train_runtime': 148.0521, 'train_samples_per_second': 9.524, 'train_steps_per_second': 0.608, 'total_flos': 120127966272000.0, 'train_loss': 0.18408127890692816, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.264300,0.407871,10,0.350000,0.066667,0.214286,0.052991,0.090476,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.000000,0.600000


{'eval_loss': 0.40787050127983093, 'eval_f1_micro': 0.35, 'eval_f1_macro': 0.06666666666666667, 'eval_accuracy': 0.21428571428571427, 'eval_precision': 0.052991452991453, 'eval_recall': 0.09047619047619047, 'eval_f1_label_0': 0.0, 'eval_f1_label_1': 0.0, 'eval_f1_label_2': 0.0, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.0, 'eval_f1_label_5': 0.0, 'eval_f1_label_6': 0.0, 'eval_f1_label_7': 0.0, 'eval_f1_label_8': 0.4, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.0, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.0, 'eval_f1_label_13': 0.0, 'eval_f1_label_14': 0.6}


## 1.3 Europe minus Germany: Volltext vs. Titel

### 1.3.1 Volltext

In [ ]:
# df_full_eur = df_full[df_full['Language']!='ge'].drop(columns='Language')

In [ ]:
# df_full_eur.head(3)

In [ ]:
# # Define the text column and extract the label columns
# text_col = 'Headline_GER'
# label_cols = [col for col in df_full_eur.columns if col != text_col]

In [ ]:
# Y = df_full_eur[label_cols].values

# # FIRST SPLIT: 80% Train, 20% Rest
# msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
# train_idx, rest_idx = next(msss1.split(df_full_eur, Y))

# train_df = df_full_eur.iloc[train_idx].copy()
# rest_df = df_full_eur.iloc[rest_idx].copy()
# Y_rest = Y[rest_idx]

# # SECOND SPLIT: Split Rest into 12% Dev / 8% Test
# msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
# dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

# dev_df = rest_df.iloc[dev_sub_idx].copy()
# test_df = rest_df.iloc[test_sub_idx].copy()

# # View
# print(f"Train size: {len(train_df)}")
# print(f"Dev size:   {len(dev_df)}")
# print(f"Test size:  {len(test_df)}")

In [ ]:
# # Convert individual binary columns into a single float list per row
# for split_df in [train_df, dev_df, test_df]:
#     split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# # Create  DatasetDict
# dataset = DatasetDict({
#     'train': Dataset.from_pandas(train_df, preserve_index=False),
#     'dev': Dataset.from_pandas(dev_df, preserve_index=False),
#     'test': Dataset.from_pandas(test_df, preserve_index=False)
# })

In [ ]:
# # Varying Hyerparameters
# tokenized_datasets = dataset.map(tokenize_function, batched=True)
# num_labels = len(label_cols)
# config.num_labels = num_labels

In [ ]:
# # Load the model with the modified configuration
# def new_model_init():
#     print("Initializing new model")
#     return AutoModelForSequenceClassification.from_pretrained(
#         model_id,
#         config=config
#     )

# trainer = Trainer(
#     model_init=new_model_init,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["dev"],
#     compute_metrics=compute_metrics
# )

In [ ]:
# # Execute the training loop
# trainer.train()

In [ ]:
# # Final evaluation against the unseen test set
# print("\n--- Final Test Set Evaluation ---")
# test_results = trainer.evaluate(tokenized_datasets["test"])
# print(test_results)

### 1.3.2 Titel

In [ ]:
# df_LLM_eur = df_LLM[df_LLM['Language']!='ge'].drop(columns=[
#     'Language',
#     # Labels
#     'Quality of life',
#     'Capacity and resources',
#     'Fairness and equality',
#     'Public opinion',
#     'Cultural identity',
#     'Policy prescription and evaluation'])

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_LLM-EURminus",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_eur_minus.columns if col != text_col]

In [ ]:
Y = df_LLM_eur_minus[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, rest_idx = next(msss1.split(df_LLM_eur_minus, Y))

train_df = df_LLM_eur_minus.iloc[train_idx].copy()
rest_df = df_LLM_eur_minus.iloc[rest_idx].copy()
Y_rest = Y[rest_idx]

# SECOND SPLIT: Split Rest into 12% Dev / 8% Test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

dev_df = rest_df.iloc[dev_sub_idx].copy()
test_df = rest_df.iloc[test_sub_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 652
Dev size:   98
Test size:  69


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/652 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
def new_model_init():
    print("Initializing new model")
    return AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    )

trainer = Trainer(
    model_init=new_model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,No log,0.204722,0.423645,0.169544,0.244898,0.357317,0.128483,0.250000,0.000000,0.142857,0.000000,0.000000,0.000000,0.307692,0.444444,0.400000,0.000000,0.000000,0.000000,0.641026,0.357143,0.000000
2,0.322112,0.157164,0.618834,0.307984,0.367347,0.477144,0.256997,0.444444,0.000000,0.285714,0.000000,0.000000,0.333333,0.625000,0.702703,0.857143,0.000000,0.000000,0.000000,0.742857,0.628571,0.000000
3,0.149961,0.137116,0.683544,0.382411,0.459184,0.485606,0.330099,0.400000,0.000000,0.588235,0.000000,0.375000,0.571429,0.800000,0.842105,0.818182,0.000000,0.000000,0.000000,0.760563,0.580645,0.000000
4,0.097829,0.126804,0.731092,0.422442,0.551020,0.549869,0.364399,0.444444,0.000000,0.818182,0.000000,0.571429,0.571429,0.842105,0.871795,0.818182,0.000000,0.000000,0.000000,0.732394,0.666667,0.000000
5,0.059119,0.122844,0.726531,0.417263,0.530612,0.519279,0.368052,0.444444,0.000000,0.800000,0.000000,0.421053,0.571429,0.842105,0.894737,0.818182,0.000000,0.000000,0.000000,0.739726,0.727273,0.000000
6,0.059119,0.122803,0.737705,0.432450,0.551020,0.527374,0.381386,0.444444,0.000000,0.800000,0.000000,0.421053,0.750000,0.842105,0.894737,0.857143,0.000000,0.000000,0.000000,0.750000,0.727273,0.000000
7,0.034750,0.121879,0.739496,0.434462,0.571429,0.549026,0.373978,0.444444,0.000000,0.800000,0.000000,0.533333,0.750000,0.842105,0.894737,0.857143,0.000000,0.000000,0.000000,0.750000,0.645161,0.000000
8,0.021386,0.123004,0.750000,0.439936,0.581633,0.551078,0.381386,0.444444,0.000000,0.800000,0.000000,0.533333,0.750000,0.842105,0.894737,0.857143,0.000000,0.000000,0.000000,0.750000,0.727273,0.000000
9,0.016050,0.123469,0.750000,0.439936,0.581633,0.551078,0.381386,0.444444,0.000000,0.800000,0.000000,0.533333,0.750000,0.842105,0.894737,0.857143,0.000000,0.000000,0.000000,0.750000,0.727273,0.000000
10,0.013925,0.123709,0.746888,0.439251,0.581633,0.549649,0.381386,0.444444,0.000000,0.800000,0.000000,0.533333,0.750000,0.842105,0.894737,0.857143,0.000000,0.000000,0.000000,0.739726,0.727273,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=410, training_loss=0.08753984995731494, metrics={'train_runtime': 294.4971, 'train_samples_per_second': 22.139, 'train_steps_per_second': 1.392, 'total_flos': 555485347584000.0, 'train_loss': 0.08753984995731494, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.013925,0.137938,10,0.696203,0.351524,0.521739,0.431154,0.320344,0.400000,0.000000,0.875000,0.000000,0.222222,0.000000,0.750000,0.692308,0.933333,0.000000,0.000000,0.000000,0.800000,0.600000,0.000000


{'eval_loss': 0.13793784379959106, 'eval_f1_micro': 0.6962025316455697, 'eval_f1_macro': 0.3515242165242165, 'eval_accuracy': 0.5217391304347826, 'eval_precision': 0.43115384615384617, 'eval_recall': 0.3203443223443223, 'eval_f1_label_0': 0.4, 'eval_f1_label_1': 0.0, 'eval_f1_label_2': 0.875, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.2222222222222222, 'eval_f1_label_5': 0.0, 'eval_f1_label_6': 0.75, 'eval_f1_label_7': 0.6923076923076923, 'eval_f1_label_8': 0.9333333333333333, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.0, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.8, 'eval_f1_label_13': 0.6, 'eval_f1_label_14': 0.0}


## 1.4 Europe plus Germany

### 1.4.1 Volltext

In [ ]:
# df_full_comp = df_full.drop(columns='Language')

In [ ]:
# df_full_comp.head(3)

In [ ]:
# # Define the text column and extract the label columns
# text_col = 'Headline_GER'
# label_cols = [col for col in df_full_comp.columns if col != text_col]

In [ ]:
# Y = df_full_comp[label_cols].values

# # FIRST SPLIT: 80% Train, 20% Rest
# msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
# train_idx, rest_idx = next(msss1.split(df_full_comp, Y))

# train_df = df_full_comp.iloc[train_idx].copy()
# rest_df = df_full_comp.iloc[rest_idx].copy()
# Y_rest = Y[rest_idx]

# # SECOND SPLIT: Split Rest into 12% Dev / 8% Test
# msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
# dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

# dev_df = rest_df.iloc[dev_sub_idx].copy()
# test_df = rest_df.iloc[test_sub_idx].copy()

# # View
# print(f"Train size: {len(train_df)}")
# print(f"Dev size:   {len(dev_df)}")
# print(f"Test size:  {len(test_df)}")

In [ ]:
# # Convert individual binary columns into a single float list per row
# for split_df in [train_df, dev_df, test_df]:
#     split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# # Create  DatasetDict
# dataset = DatasetDict({
#     'train': Dataset.from_pandas(train_df, preserve_index=False),
#     'dev': Dataset.from_pandas(dev_df, preserve_index=False),
#     'test': Dataset.from_pandas(test_df, preserve_index=False)
# })

In [ ]:
# # Varying Hyerparameters
# tokenized_datasets = dataset.map(tokenize_function, batched=True)
# num_labels = len(label_cols)
# config.num_labels = num_labels

In [ ]:
# # Load the model with the modified configuration
# def new_model_init():
#     print("Initializing new model")
#     return AutoModelForSequenceClassification.from_pretrained(
#         model_id,
#         config=config
#     )

# trainer = Trainer(
#     model_init=new_model_init,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["dev"],
#     compute_metrics=compute_metrics
# )

In [ ]:
# # Execute the training loop
# trainer.train()

In [ ]:
# # Final evaluation against the unseen test set
# print("\n--- Final Test Set Evaluation ---")
# test_results = trainer.evaluate(tokenized_datasets["test"])
# print(test_results)

### 1.4.2 Titel

In [ ]:
# df_LLM_comp = df_LLM.drop(columns=[
#     'Language',
#     # Labels
#     'Quality of life',
#     'Capacity and resources',
#     'Fairness and equality',
#     'Public opinion',
#     'Cultural identity',
#     'Policy prescription and evaluation'])

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_LLM-EURplus",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_eur_plus.columns if col != text_col]

In [ ]:
Y = df_LLM_eur_plus[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, rest_idx = next(msss1.split(df_LLM_eur_plus, Y))

train_df = df_LLM_eur_plus.iloc[train_idx].copy()
rest_df = df_LLM_eur_plus.iloc[rest_idx].copy()
Y_rest = Y[rest_idx]

# SECOND SPLIT: Split Rest into 12% Dev / 8% Test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

dev_df = rest_df.iloc[dev_sub_idx].copy()
test_df = rest_df.iloc[test_sub_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 791
Dev size:   124
Test size:  81


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/791 [00:00<?, ? examples/s]

Map:   0%|          | 0/124 [00:00<?, ? examples/s]

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
def new_model_init():
    print("Initializing new model")
    return AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    )

trainer = Trainer(
    model_init=new_model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,0.327028,0.207078,0.387931,0.163671,0.201613,0.221645,0.133627,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.250000,0.400000,0.727273,0.000000,0.000000,0.000000,0.548387,0.529412,0.000000
2,0.172136,0.166161,0.586466,0.314478,0.354839,0.430198,0.269848,0.666667,0.000000,0.666667,0.000000,0.000000,0.000000,0.400000,0.590909,0.880000,0.000000,0.000000,0.000000,0.720000,0.611111,0.181818
3,0.121401,0.153688,0.640845,0.346896,0.419355,0.414242,0.315774,0.500000,0.000000,0.666667,0.000000,0.000000,0.000000,0.608696,0.625000,0.880000,0.000000,0.000000,0.000000,0.794872,0.666667,0.461538
4,0.081755,0.147199,0.669065,0.376822,0.475806,0.461186,0.332210,0.769231,0.000000,0.818182,0.000000,0.000000,0.000000,0.600000,0.651163,0.923077,0.000000,0.000000,0.000000,0.780488,0.648649,0.461538
5,0.052296,0.143871,0.671480,0.389347,0.491935,0.493315,0.336604,0.666667,0.000000,0.782609,0.000000,0.153846,0.000000,0.600000,0.651163,0.923077,0.000000,0.000000,0.000000,0.790123,0.606061,0.666667
6,0.032122,0.141318,0.702128,0.422325,0.516129,0.522667,0.364914,0.666667,0.000000,0.869565,0.000000,0.500000,0.000000,0.600000,0.666667,0.923077,0.000000,0.000000,0.000000,0.795181,0.647059,0.666667
7,0.021749,0.143715,0.673684,0.391417,0.508065,0.487143,0.344937,0.666667,0.000000,0.750000,0.000000,0.153846,0.000000,0.600000,0.666667,0.923077,0.000000,0.000000,0.000000,0.758621,0.685714,0.666667
8,0.016611,0.143323,0.697842,0.410847,0.508065,0.516296,0.353775,0.769231,0.000000,0.818182,0.000000,0.285714,0.000000,0.526316,0.682927,0.923077,0.000000,0.000000,0.000000,0.804878,0.685714,0.666667
9,0.013906,0.144505,0.690647,0.408779,0.500000,0.513219,0.352109,0.769231,0.000000,0.818182,0.000000,0.285714,0.000000,0.526316,0.666667,0.923077,0.000000,0.000000,0.000000,0.790123,0.685714,0.666667
10,0.012766,0.144947,0.688172,0.408594,0.500000,0.513528,0.352109,0.769231,0.000000,0.818182,0.000000,0.285714,0.000000,0.526316,0.682927,0.923077,0.000000,0.000000,0.000000,0.771084,0.685714,0.666667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.08517691004276276, metrics={'train_runtime': 318.8129, 'train_samples_per_second': 24.811, 'train_steps_per_second': 1.568, 'total_flos': 673909371072000.0, 'train_loss': 0.08517691004276276, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.012766,0.145197,10,0.648045,0.377317,0.419753,0.491932,0.319408,0.500000,0.000000,0.857143,0.000000,0.444444,0.000000,0.769231,0.642857,0.823529,0.000000,0.000000,0.000000,0.705882,0.666667,0.250000


{'eval_loss': 0.14519715309143066, 'eval_f1_micro': 0.6480446927374302, 'eval_f1_macro': 0.3773169096698509, 'eval_accuracy': 0.41975308641975306, 'eval_precision': 0.49193162393162393, 'eval_recall': 0.3194078144078144, 'eval_f1_label_0': 0.5, 'eval_f1_label_1': 0.0, 'eval_f1_label_2': 0.8571428571428571, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.4444444444444444, 'eval_f1_label_5': 0.0, 'eval_f1_label_6': 0.7692307692307693, 'eval_f1_label_7': 0.6428571428571429, 'eval_f1_label_8': 0.8235294117647058, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.0, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.7058823529411765, 'eval_f1_label_13': 0.6666666666666666, 'eval_f1_label_14': 0.25}


## 1.5 Europe & MFC

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_LLM-MFC",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_eur_mfc.columns if col != text_col]

In [ ]:
Y = df_LLM_eur_mfc[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, rest_idx = next(msss1.split(df_LLM_eur_mfc, Y))

train_df = df_LLM_eur_mfc.iloc[train_idx].copy()
rest_df = df_LLM_eur_mfc.iloc[rest_idx].copy()
Y_rest = Y[rest_idx]

# SECOND SPLIT: Split Rest into 12% Dev / 8% Test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

dev_df = rest_df.iloc[dev_sub_idx].copy()
test_df = rest_df.iloc[test_sub_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 1818
Dev size:   274
Test size:  187


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/1818 [00:00<?, ? examples/s]

Map:   0%|          | 0/274 [00:00<?, ? examples/s]

Map:   0%|          | 0/187 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
def new_model_init():
    print("Initializing new model")
    return AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    )

trainer = Trainer(
    model_init=new_model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,0.177151,0.148760,0.546939,0.313798,0.401460,0.512631,0.274526,0.230769,0.714286,0.125000,0.000000,0.697248,0.710280,0.117647,0.424242,0.812500,0.000000,0.000000,0.000000,0.500000,0.375000,0.000000
2,0.113241,0.125721,0.671727,0.465198,0.507299,0.596875,0.409804,0.628571,0.740741,0.857143,0.333333,0.796610,0.785047,0.370370,0.594595,0.866667,0.000000,0.000000,0.000000,0.580645,0.424242,0.000000
3,0.067831,0.115772,0.719023,0.500994,0.583942,0.571637,0.472405,0.588235,0.785714,0.846154,0.285714,0.806723,0.827586,0.454545,0.744186,0.866667,0.000000,0.000000,0.000000,0.676056,0.633333,0.000000
4,0.040468,0.111896,0.725352,0.515161,0.594891,0.578353,0.476996,0.685714,0.814815,0.888889,0.500000,0.850000,0.823529,0.434783,0.634146,0.866667,0.000000,0.000000,0.000000,0.675676,0.553191,0.000000
5,0.018060,0.113658,0.726316,0.505695,0.609489,0.566609,0.469810,0.722222,0.814815,0.846154,0.285714,0.819672,0.823529,0.416667,0.727273,0.866667,0.000000,0.000000,0.000000,0.684932,0.577778,0.000000
6,0.010438,0.117215,0.721831,0.506432,0.602190,0.565406,0.471691,0.722222,0.814815,0.846154,0.285714,0.800000,0.820513,0.416667,0.727273,0.866667,0.000000,0.000000,0.000000,0.684211,0.612245,0.000000
7,0.007289,0.119291,0.728242,0.512505,0.605839,0.605870,0.470308,0.722222,0.814815,0.846154,0.333333,0.800000,0.813559,0.416667,0.727273,0.866667,0.000000,0.000000,0.000000,0.710526,0.636364,0.000000
8,0.005746,0.122555,0.716312,0.505391,0.594891,0.598108,0.465883,0.722222,0.814815,0.846154,0.333333,0.800000,0.813559,0.416667,0.697674,0.866667,0.000000,0.000000,0.000000,0.657534,0.612245,0.000000
9,0.004532,0.123520,0.717857,0.485639,0.594891,0.533496,0.452429,0.722222,0.814815,0.846154,0.000000,0.800000,0.813559,0.400000,0.727273,0.866667,0.000000,0.000000,0.000000,0.657534,0.636364,0.000000
10,0.004357,0.123864,0.717857,0.485639,0.594891,0.533496,0.452429,0.722222,0.814815,0.846154,0.000000,0.800000,0.813559,0.400000,0.727273,0.866667,0.000000,0.000000,0.000000,0.657534,0.636364,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1140, training_loss=0.0508725603813665, metrics={'train_runtime': 624.016, 'train_samples_per_second': 29.134, 'train_steps_per_second': 1.827, 'total_flos': 1548883990656000.0, 'train_loss': 0.0508725603813665, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.004357,0.106829,10,0.761658,0.529014,0.647059,0.652269,0.474177,0.769231,0.842105,0.750000,0.000000,0.800000,0.785714,0.533333,0.814815,0.750000,0.000000,0.000000,0.000000,0.870968,0.733333,0.285714


{'eval_loss': 0.10682888329029083, 'eval_f1_micro': 0.7616580310880829, 'eval_f1_macro': 0.5290142551489467, 'eval_accuracy': 0.6470588235294118, 'eval_precision': 0.652269296607532, 'eval_recall': 0.4741773132017035, 'eval_f1_label_0': 0.7692307692307693, 'eval_f1_label_1': 0.8421052631578947, 'eval_f1_label_2': 0.75, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.8, 'eval_f1_label_5': 0.7857142857142857, 'eval_f1_label_6': 0.5333333333333333, 'eval_f1_label_7': 0.8148148148148148, 'eval_f1_label_8': 0.75, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.0, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.8709677419354839, 'eval_f1_label_13': 0.7333333333333333, 'eval_f1_label_14': 0.2857142857142857}


## 1.6 Europe & CHN

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_LLM-CHN",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_eur_chn.columns if col != text_col]

In [ ]:
Y = df_LLM_eur_chn[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, rest_idx = next(msss1.split(df_LLM_eur_chn, Y))

train_df = df_LLM_eur_chn.iloc[train_idx].copy()
rest_df = df_LLM_eur_chn.iloc[rest_idx].copy()
Y_rest = Y[rest_idx]

# SECOND SPLIT: Split Rest into 12% Dev / 8% Test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

dev_df = rest_df.iloc[dev_sub_idx].copy()
test_df = rest_df.iloc[test_sub_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 1032
Dev size:   166
Test size:  109


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/1032 [00:00<?, ? examples/s]

Map:   0%|          | 0/166 [00:00<?, ? examples/s]

Map:   0%|          | 0/109 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
def new_model_init():
    print("Initializing new model")
    return AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    )

trainer = Trainer(
    model_init=new_model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,0.346709,0.210117,0.377483,0.186451,0.192771,0.358323,0.139928,0.352941,0.000000,0.133333,0.000000,0.000000,0.000000,0.320000,0.270270,0.600000,0.000000,0.000000,0.000000,0.558824,0.561404,0.000000
2,0.193270,0.182792,0.553425,0.321731,0.343373,0.498765,0.273940,0.526316,0.000000,0.526316,0.000000,0.000000,0.000000,0.606061,0.509804,0.709677,0.000000,0.400000,0.000000,0.727273,0.666667,0.153846
3,0.146760,0.175368,0.559783,0.354170,0.349398,0.456348,0.306598,0.608696,0.000000,0.695652,0.000000,0.117647,0.235294,0.705882,0.428571,0.685714,0.000000,0.333333,0.000000,0.717391,0.641509,0.142857
4,0.074141,0.170331,0.586387,0.383197,0.373494,0.467622,0.337188,0.608696,0.000000,0.750000,0.000000,0.125000,0.333333,0.687500,0.510638,0.685714,0.000000,0.285714,0.000000,0.731183,0.655172,0.375000
5,0.049740,0.173802,0.612658,0.414629,0.379518,0.482415,0.382017,0.695652,0.000000,0.689655,0.000000,0.272727,0.545455,0.764706,0.530612,0.722222,0.000000,0.333333,0.000000,0.755556,0.642857,0.266667
6,0.035309,0.177752,0.602094,0.391921,0.373494,0.485355,0.348368,0.727273,0.000000,0.800000,0.000000,0.125000,0.421053,0.764706,0.521739,0.685714,0.000000,0.333333,0.000000,0.750000,0.607143,0.142857
7,0.017273,0.179971,0.600522,0.398284,0.373494,0.482544,0.353771,0.666667,0.000000,0.740741,0.000000,0.222222,0.421053,0.764706,0.521739,0.666667,0.000000,0.333333,0.000000,0.755556,0.631579,0.250000
8,0.013558,0.182712,0.606383,0.400420,0.391566,0.509728,0.350053,0.666667,0.000000,0.800000,0.000000,0.133333,0.444444,0.800000,0.521739,0.685714,0.000000,0.333333,0.000000,0.747253,0.607143,0.266667
9,0.012244,0.184264,0.608466,0.400772,0.385542,0.504556,0.351569,0.666667,0.000000,0.800000,0.000000,0.125000,0.444444,0.800000,0.521739,0.685714,0.000000,0.333333,0.000000,0.760870,0.607143,0.266667
10,0.010512,0.184907,0.606383,0.400600,0.391566,0.506225,0.350053,0.666667,0.000000,0.800000,0.000000,0.125000,0.444444,0.800000,0.521739,0.685714,0.000000,0.333333,0.000000,0.747253,0.618182,0.266667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=650, training_loss=0.08043755879768959, metrics={'train_runtime': 391.0005, 'train_samples_per_second': 26.394, 'train_steps_per_second': 1.662, 'total_flos': 879234476544000.0, 'train_loss': 0.08043755879768959, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.010512,0.164693,10,0.656250,0.402767,0.385321,0.541805,0.361783,0.461538,0.000000,0.705882,0.000000,0.461538,0.222222,0.700000,0.790698,0.928571,0.000000,0.000000,0.000000,0.734694,0.636364,0.400000


{'eval_loss': 0.16469259560108185, 'eval_f1_micro': 0.65625, 'eval_f1_macro': 0.40276720767633417, 'eval_accuracy': 0.3853211009174312, 'eval_precision': 0.5418054248489031, 'eval_recall': 0.361783350748868, 'eval_f1_label_0': 0.46153846153846156, 'eval_f1_label_1': 0.0, 'eval_f1_label_2': 0.7058823529411765, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.46153846153846156, 'eval_f1_label_5': 0.2222222222222222, 'eval_f1_label_6': 0.7, 'eval_f1_label_7': 0.7906976744186046, 'eval_f1_label_8': 0.9285714285714286, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.0, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.7346938775510204, 'eval_f1_label_13': 0.6363636363636364, 'eval_f1_label_14': 0.4}


## 1.7 Complete Data

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_LLM-comp",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_comp.columns if col != text_col]

In [ ]:
Y = df_LLM_comp[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, rest_idx = next(msss1.split(df_LLM_comp, Y))

train_df = df_LLM_comp.iloc[train_idx].copy()
rest_df = df_LLM_comp.iloc[rest_idx].copy()
Y_rest = Y[rest_idx]

# SECOND SPLIT: Split Rest into 12% Dev / 8% Test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

dev_df = rest_df.iloc[dev_sub_idx].copy()
test_df = rest_df.iloc[test_sub_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 2065
Dev size:   320
Test size:  205


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/2065 [00:00<?, ? examples/s]

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/205 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
def new_model_init():
    print("Initializing new model")
    return AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    )

trainer = Trainer(
    model_init=new_model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Initializing new model


model.safetensors: reconstructing file:   0%|          |  0.00B /  635MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Initializing new model


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,0.191809,0.157798,0.565495,0.377290,0.390625,0.475878,0.323362,0.526316,0.518519,0.416667,0.000000,0.666667,0.677966,0.451613,0.372093,0.882353,0.000000,0.000000,0.000000,0.580000,0.567164,0.000000
2,0.115257,0.132169,0.671733,0.461382,0.515625,0.525605,0.420226,0.711111,0.687500,0.689655,0.000000,0.800000,0.776860,0.533333,0.600000,0.914286,0.000000,0.000000,0.000000,0.623377,0.584615,0.000000
3,0.075467,0.129639,0.694286,0.501598,0.543750,0.580252,0.470454,0.755556,0.733333,0.709677,0.000000,0.823529,0.790323,0.700000,0.541667,0.882353,0.000000,0.000000,0.333333,0.654206,0.600000,0.000000
4,0.040361,0.128065,0.702782,0.510172,0.553125,0.587964,0.473863,0.808511,0.709677,0.709677,0.000000,0.818898,0.803150,0.684211,0.655738,0.914286,0.000000,0.000000,0.333333,0.643678,0.571429,0.000000
5,0.018630,0.135189,0.706056,0.507587,0.571875,0.604189,0.460894,0.808511,0.733333,0.642857,0.000000,0.818182,0.806723,0.685714,0.654545,0.882353,0.000000,0.000000,0.333333,0.705882,0.542373,0.000000
6,0.011220,0.138187,0.709677,0.508360,0.578125,0.597188,0.465751,0.808511,0.709677,0.642857,0.000000,0.812030,0.818898,0.685714,0.654545,0.857143,0.000000,0.000000,0.333333,0.696629,0.606061,0.000000
7,0.007488,0.140751,0.697329,0.500957,0.559375,0.595167,0.454427,0.782609,0.709677,0.642857,0.000000,0.803030,0.803279,0.685714,0.629630,0.857143,0.000000,0.000000,0.333333,0.695652,0.571429,0.000000
8,0.006144,0.143923,0.697329,0.499911,0.562500,0.597288,0.452502,0.782609,0.709677,0.642857,0.000000,0.812030,0.793388,0.666667,0.618182,0.857143,0.000000,0.000000,0.333333,0.702128,0.580645,0.000000
9,0.005246,0.144345,0.702065,0.502377,0.568750,0.594033,0.457856,0.782609,0.709677,0.642857,0.000000,0.812030,0.813008,0.685714,0.618182,0.857143,0.000000,0.000000,0.333333,0.709677,0.571429,0.000000
10,0.004802,0.145185,0.706056,0.504927,0.568750,0.599644,0.458914,0.782609,0.709677,0.642857,0.000000,0.820896,0.813008,0.705882,0.618182,0.857143,0.000000,0.000000,0.333333,0.709677,0.580645,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1300, training_loss=0.05277860501637826, metrics={'train_runtime': 800.0738, 'train_samples_per_second': 25.81, 'train_steps_per_second': 1.625, 'total_flos': 1759320924480000.0, 'train_loss': 0.05277860501637826, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.004802,0.125393,10,0.715556,0.532112,0.570732,0.624349,0.487532,0.758621,0.777778,0.818182,0.000000,0.864198,0.790698,0.583333,0.744186,0.720000,0.000000,0.500000,0.000000,0.711864,0.512821,0.200000


{'eval_loss': 0.1253933161497116, 'eval_f1_micro': 0.7155555555555555, 'eval_f1_macro': 0.532111986022847, 'eval_accuracy': 0.5707317073170731, 'eval_precision': 0.6243492095666009, 'eval_recall': 0.4875316588072343, 'eval_f1_label_0': 0.7586206896551724, 'eval_f1_label_1': 0.7777777777777778, 'eval_f1_label_2': 0.8181818181818182, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.8641975308641975, 'eval_f1_label_5': 0.7906976744186046, 'eval_f1_label_6': 0.5833333333333334, 'eval_f1_label_7': 0.7441860465116279, 'eval_f1_label_8': 0.72, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.5, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.711864406779661, 'eval_f1_label_13': 0.5128205128205128, 'eval_f1_label_14': 0.2}


In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)

Best checkpoint: ./modern_gbert_finetuned_LLM-comp/checkpoint-520
Best metric: 0.5101724488202012


In [ ]:
# login to huggingface
from huggingface_hub import login

login()

In [ ]:
# Save the best model and tokenizer to hugginface
trainer.model.push_to_hub("GER_frame-detect_mgbert134M")
tokenizer.push_to_hub("GER_frame-detect_mgbert134M")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...g_rg4rg/model.safetensors:   0%|          |  556kB /  539MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/nameless-0/GER_frame-detect_mgbert134M/commit/b54c545326f3639bf431941778e55429ad6db5b7', commit_message='Upload tokenizer', commit_description='', oid='b54c545326f3639bf431941778e55429ad6db5b7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/nameless-0/GER_frame-detect_mgbert134M', endpoint='https://huggingface.co', repo_type='model', repo_id='nameless-0/GER_frame-detect_mgbert134M'), pr_revision=None, pr_num=None)

# 2 Approximate Transfer

## 2.1 Train mit ALL minus GER & CHN & MFC (=EU minus GER)

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_Transfer-AllMin_GER-MFC-CHN",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_ex_GerChnMFC.columns if col != text_col]

In [ ]:
Y_allmin_GerChnMFC = df_LLM_ex_GerChnMFC[label_cols].values
Y_ger = df_LLM_ger[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
## COMP
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_eur_idx, rest_eur_idx = next(msss1.split(df_LLM_ex_GerChnMFC, Y_allmin_GerChnMFC))
## GERMANY
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_ger_idx, test_ger_idx = next(msss1.split(df_LLM_ger, Y_ger))

train_df = df_LLM_ex_GerChnMFC.iloc[train_eur_idx].copy()
dev_df = df_LLM_ger.iloc[dev_ger_idx].copy()
test_df = df_LLM_ger.iloc[test_ger_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 652
Dev size:   106
Test size:  71


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/652 [00:00<?, ? examples/s]

Map:   0%|          | 0/106 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
trainer = Trainer(
    model_init=lambda: AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    ),
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,No log,0.343025,0.229885,0.110895,0.132075,0.126943,0.109972,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.160000,0.444444,0.000000,0.000000,0.000000,0.358974,0.700000,0.000000
2,0.322112,0.320779,0.375635,0.213352,0.198113,0.234596,0.231058,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.375000,0.758621,0.000000,0.000000,0.000000,0.400000,0.666667,0.000000
3,0.149961,0.333379,0.381910,0.219321,0.226415,0.234564,0.239157,0.533333,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.424242,0.733333,0.000000,0.000000,0.000000,0.384615,0.714286,0.000000
4,0.097829,0.331557,0.410000,0.228571,0.226415,0.220327,0.254542,0.615385,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.437500,0.709677,0.000000,0.000000,0.000000,0.551724,0.714286,0.000000
5,0.059119,0.329081,0.417910,0.230724,0.235849,0.229074,0.253850,0.545455,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.470588,0.758621,0.000000,0.000000,0.000000,0.545455,0.740741,0.000000
6,0.059119,0.326802,0.395833,0.220560,0.216981,0.220042,0.234762,0.500000,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.437500,0.785714,0.000000,0.000000,0.000000,0.444444,0.740741,0.000000
7,0.034750,0.322444,0.422680,0.234183,0.245283,0.240353,0.247789,0.545455,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.516129,0.785714,0.000000,0.000000,0.000000,0.545455,0.720000,0.000000
8,0.021386,0.322514,0.432161,0.234558,0.254717,0.227216,0.257554,0.500000,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.529412,0.785714,0.000000,0.000000,0.000000,0.562500,0.740741,0.000000
9,0.016050,0.320802,0.432161,0.234558,0.254717,0.227216,0.257554,0.500000,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.529412,0.785714,0.000000,0.000000,0.000000,0.562500,0.740741,0.000000
10,0.013925,0.321236,0.432161,0.236452,0.254717,0.235637,0.257554,0.545455,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.529412,0.785714,0.000000,0.000000,0.000000,0.545455,0.740741,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=410, training_loss=0.08753984995731494, metrics={'train_runtime': 268.43, 'train_samples_per_second': 24.289, 'train_steps_per_second': 1.527, 'total_flos': 555485347584000.0, 'train_loss': 0.08753984995731494, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.013925,0.343424,10,0.379562,0.208687,0.253521,0.193290,0.232407,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.666667,0.700000,0.000000,0.000000,0.000000,0.400000,0.363636,0.000000


{'eval_loss': 0.3434238135814667, 'eval_f1_micro': 0.3795620437956204, 'eval_f1_macro': 0.20868686868686867, 'eval_accuracy': 0.2535211267605634, 'eval_precision': 0.1932900432900433, 'eval_recall': 0.23240740740740737, 'eval_f1_label_0': 0.5, 'eval_f1_label_1': 0.0, 'eval_f1_label_2': 0.5, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.0, 'eval_f1_label_5': 0.0, 'eval_f1_label_6': 0.0, 'eval_f1_label_7': 0.6666666666666666, 'eval_f1_label_8': 0.7, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.0, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.4, 'eval_f1_label_13': 0.36363636363636365, 'eval_f1_label_14': 0.0}


## 2.2 Train mit ALL minus GER & MFC

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_Transfer-AllMin_GER-MFC",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_ex_GerMFC.columns if col != text_col]

In [ ]:
Y_allmin_GerMFC = df_LLM_ex_GerMFC[label_cols].values
Y_ger = df_LLM_ger[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
## COMP
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_eur_idx, rest_eur_idx = next(msss1.split(df_LLM_ex_GerMFC, Y_allmin_GerMFC))
## GERMANY
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_ger_idx, test_ger_idx = next(msss1.split(df_LLM_ger, Y_ger))

train_df = df_LLM_ex_GerMFC.iloc[train_eur_idx].copy()
dev_df = df_LLM_ger.iloc[dev_ger_idx].copy()
test_df = df_LLM_ger.iloc[test_ger_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 894
Dev size:   106
Test size:  71


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Map:   0%|          | 0/106 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
trainer = Trainer(
    model_init=lambda: AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    ),
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,0.337543,0.306359,0.292135,0.147095,0.160377,0.165360,0.146457,0.222222,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.333333,0.750000,0.000000,0.000000,0.000000,0.260870,0.640000,0.000000
2,0.182084,0.285217,0.364532,0.208532,0.198113,0.224366,0.235946,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.357143,0.733333,0.000000,0.000000,0.000000,0.437500,0.600000,0.000000
3,0.131253,0.282559,0.378109,0.217885,0.216981,0.236181,0.239157,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.451613,0.733333,0.000000,0.000000,0.000000,0.416667,0.666667,0.000000
4,0.089642,0.275499,0.392523,0.223602,0.254717,0.231373,0.258245,0.533333,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.457143,0.666667,0.000000,0.000000,0.000000,0.551724,0.645161,0.000000
5,0.065802,0.294875,0.388626,0.217447,0.226415,0.224377,0.254542,0.421053,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.424242,0.758621,0.000000,0.000000,0.000000,0.551724,0.606061,0.000000
6,0.040404,0.283440,0.398058,0.226775,0.245283,0.237037,0.252185,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.470588,0.758621,0.000000,0.000000,0.000000,0.551724,0.620690,0.000000
7,0.027285,0.293589,0.405797,0.224703,0.235849,0.231650,0.258245,0.444444,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.470588,0.758621,0.000000,0.000000,0.000000,0.551724,0.645161,0.000000
8,0.019981,0.297215,0.409756,0.226446,0.235849,0.234074,0.258245,0.470588,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.470588,0.758621,0.000000,0.000000,0.000000,0.551724,0.645161,0.000000
9,0.013258,0.298105,0.401961,0.224431,0.226415,0.233606,0.253117,0.470588,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.470588,0.758621,0.000000,0.000000,0.000000,0.500000,0.666667,0.000000
10,0.012281,0.299827,0.411765,0.227879,0.235849,0.235828,0.258245,0.470588,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.470588,0.758621,0.000000,0.000000,0.000000,0.551724,0.666667,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=560, training_loss=0.0837097958794662, metrics={'train_runtime': 352.2313, 'train_samples_per_second': 25.381, 'train_steps_per_second': 1.59, 'total_flos': 761662424448000.0, 'train_loss': 0.0837097958794662, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.012281,0.316776,10,0.419580,0.235004,0.281690,0.231429,0.262037,0.444444,0.000000,0.666667,0.000000,0.000000,0.000000,0.000000,0.692308,0.700000,0.000000,0.000000,0.000000,0.476190,0.545455,0.000000


{'eval_loss': 0.31677576899528503, 'eval_f1_micro': 0.4195804195804196, 'eval_f1_macro': 0.235004255004255, 'eval_accuracy': 0.28169014084507044, 'eval_precision': 0.2314285714285714, 'eval_recall': 0.262037037037037, 'eval_f1_label_0': 0.4444444444444444, 'eval_f1_label_1': 0.0, 'eval_f1_label_2': 0.6666666666666666, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.0, 'eval_f1_label_5': 0.0, 'eval_f1_label_6': 0.0, 'eval_f1_label_7': 0.6923076923076923, 'eval_f1_label_8': 0.7, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.0, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.47619047619047616, 'eval_f1_label_13': 0.5454545454545454, 'eval_f1_label_14': 0.0}


## 2.3 Train mit ALL minus GER & CHN

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_Transfer-AllMin_GER-CHN",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_ex_GerChn.columns if col != text_col]

In [ ]:
Y_allmin_GerChn = df_LLM_ex_GerChn[label_cols].values
Y_ger = df_LLM_ger[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
## COMP
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_eur_idx, rest_eur_idx = next(msss1.split(df_LLM_ex_GerChn, Y_allmin_GerChn))
## GERMANY
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_ger_idx, test_ger_idx = next(msss1.split(df_LLM_ger, Y_ger))

train_df = df_LLM_ex_GerChn.iloc[train_eur_idx].copy()
dev_df = df_LLM_ger.iloc[dev_ger_idx].copy()
test_df = df_LLM_ger.iloc[test_ger_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 1676
Dev size:   106
Test size:  71


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/1676 [00:00<?, ? examples/s]

Map:   0%|          | 0/106 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
trainer = Trainer(
    model_init=lambda: AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    ),
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,0.166238,0.321755,0.320000,0.164133,0.198113,0.177691,0.164399,0.307692,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.320000,0.846154,0.000000,0.000000,0.000000,0.260870,0.727273,0.000000
2,0.104643,0.328166,0.369231,0.209668,0.207547,0.225767,0.228901,0.444444,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.400000,0.758621,0.000000,0.000000,0.000000,0.272727,0.769231,0.000000
3,0.066357,0.341475,0.401961,0.255549,0.245283,0.266639,0.281322,0.444444,0.000000,0.500000,0.000000,0.000000,0.000000,0.500000,0.484848,0.785714,0.000000,0.000000,0.000000,0.428571,0.689655,0.000000
4,0.033439,0.336768,0.388350,0.213952,0.216981,0.198941,0.248481,0.400000,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.451613,0.814815,0.000000,0.000000,0.000000,0.500000,0.642857,0.000000
5,0.018002,0.335475,0.390000,0.264673,0.254717,0.302289,0.270626,0.444444,0.000000,0.500000,0.000000,0.000000,0.000000,0.666667,0.437500,0.814815,0.000000,0.000000,0.000000,0.466667,0.640000,0.000000
6,0.010391,0.335580,0.380000,0.214509,0.226415,0.203409,0.238225,0.444444,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.437500,0.814815,0.000000,0.000000,0.000000,0.428571,0.692308,0.000000
7,0.006561,0.339658,0.385787,0.221145,0.235849,0.239509,0.234762,0.352941,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.451613,0.814815,0.000000,0.000000,0.000000,0.428571,0.769231,0.000000
8,0.005347,0.339972,0.387755,0.223088,0.245283,0.243416,0.232405,0.352941,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.814815,0.000000,0.000000,0.000000,0.428571,0.750000,0.000000
9,0.004752,0.341764,0.383838,0.221184,0.245283,0.240278,0.232405,0.352941,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.814815,0.000000,0.000000,0.000000,0.400000,0.750000,0.000000
10,0.004242,0.342165,0.387755,0.223088,0.245283,0.243416,0.232405,0.352941,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.814815,0.000000,0.000000,0.000000,0.428571,0.750000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1050, training_loss=0.049359238303843, metrics={'train_runtime': 618.3473, 'train_samples_per_second': 27.105, 'train_steps_per_second': 1.698, 'total_flos': 1427904052992000.0, 'train_loss': 0.049359238303843, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.004242,0.362359,10,0.394366,0.232366,0.267606,0.237896,0.245370,0.500000,0.000000,0.666667,0.000000,0.000000,0.000000,0.000000,0.720000,0.777778,0.000000,0.000000,0.000000,0.400000,0.421053,0.000000


{'eval_loss': 0.36235910654067993, 'eval_f1_micro': 0.39436619718309857, 'eval_f1_macro': 0.23236647173489278, 'eval_accuracy': 0.2676056338028169, 'eval_precision': 0.23789627039627043, 'eval_recall': 0.24537037037037035, 'eval_f1_label_0': 0.5, 'eval_f1_label_1': 0.0, 'eval_f1_label_2': 0.6666666666666666, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.0, 'eval_f1_label_5': 0.0, 'eval_f1_label_6': 0.0, 'eval_f1_label_7': 0.72, 'eval_f1_label_8': 0.7777777777777778, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.0, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.4, 'eval_f1_label_13': 0.42105263157894735, 'eval_f1_label_14': 0.0}


## 2.4 Train mit All minus GER, Test auf Germany

In [ ]:
# Application-specific arguments
application_training_args = {
    "output_dir": f"./modern_gbert_finetuned_Transfer-AllMin_GER",
}

# Combine them
training_args = TrainingArguments(
    **general_training_args,
    **application_training_args,
)

In [ ]:
# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_ex_Ger.columns if col != text_col]

In [ ]:
Y_allmin_Ger = df_LLM_ex_Ger[label_cols].values
Y_ger = df_LLM_ger[label_cols].values

# FIRST SPLIT: 80% Train, 20% Rest
## COMP
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_eur_idx, rest_eur_idx = next(msss1.split(df_LLM_ex_Ger, Y_allmin_Ger))
## GERMANY
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
dev_ger_idx, test_ger_idx = next(msss1.split(df_LLM_ger, Y_ger))

train_df = df_LLM_ex_Ger.iloc[train_eur_idx].copy()
dev_df = df_LLM_ger.iloc[dev_ger_idx].copy()
test_df = df_LLM_ger.iloc[test_ger_idx].copy()


# rest_df = df_full_ger.iloc[rest_ger_idx].copy()
# Y_rest = Y_ger[rest_ger_idx]

# SECOND SPLIT on GERMANY only: Split Rest into 12% Dev / 8% Test
#msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=42)
#dev_sub_idx, test_sub_idx = next(msss2.split(rest_df, Y_rest))

#dev_df = rest_df.iloc[dev_sub_idx].copy()
#test_df = rest_df.iloc[test_sub_idx].copy()

# View
print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 1925
Dev size:   106
Test size:  71


In [ ]:
# Convert individual binary columns into a single float list per row
for split_df in [train_df, dev_df, test_df]:
    split_df['labels'] = split_df[label_cols].astype(float).values.tolist()

In [ ]:
# Create  DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# Varying Hyerparameters
tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = len(label_cols)
config.num_labels = num_labels

Map:   0%|          | 0/1925 [00:00<?, ? examples/s]

Map:   0%|          | 0/106 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

In [ ]:
# Load the model with the modified configuration
trainer = Trainer(
    model_init=lambda: AutoModelForSequenceClassification.from_pretrained(
        model_id,
        config=config
    ),
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Execute the training loop
trainer.train()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: LSX-UniWue/ModernGBERT_134M
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
1,0.178508,0.292534,0.340206,0.191685,0.198113,0.217071,0.206149,0.307692,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.400000,0.846154,0.000000,0.000000,0.000000,0.250000,0.571429,0.000000
2,0.110221,0.285733,0.357143,0.202305,0.188679,0.224930,0.226622,0.400000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.370370,0.785714,0.000000,0.000000,0.000000,0.333333,0.645161,0.000000
3,0.066999,0.289682,0.382775,0.210099,0.216981,0.220755,0.250838,0.400000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.413793,0.733333,0.000000,0.000000,0.000000,0.516129,0.588235,0.000000
4,0.036154,0.301235,0.386473,0.215186,0.235849,0.225185,0.247057,0.363636,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.484848,0.758621,0.000000,0.000000,0.000000,0.500000,0.620690,0.000000
5,0.019843,0.308086,0.370000,0.209509,0.216981,0.225189,0.233097,0.400000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.451613,0.785714,0.000000,0.000000,0.000000,0.384615,0.620690,0.000000
6,0.011788,0.311346,0.396040,0.221192,0.235849,0.233781,0.247989,0.421053,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.785714,0.000000,0.000000,0.000000,0.444444,0.666667,0.000000
7,0.007122,0.316820,0.375635,0.213798,0.226415,0.229814,0.231672,0.444444,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.484848,0.814815,0.000000,0.000000,0.000000,0.320000,0.642857,0.000000
8,0.005606,0.319506,0.365482,0.208507,0.226415,0.225157,0.225611,0.444444,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.484848,0.785714,0.000000,0.000000,0.000000,0.320000,0.592593,0.000000
9,0.004934,0.323962,0.375635,0.212868,0.226415,0.229658,0.231672,0.444444,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.785714,0.000000,0.000000,0.000000,0.320000,0.642857,0.000000
10,0.004612,0.324891,0.375635,0.212868,0.226415,0.229658,0.231672,0.444444,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.785714,0.000000,0.000000,0.000000,0.320000,0.642857,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1210, training_loss=0.051422107435208704, metrics={'train_runtime': 692.9493, 'train_samples_per_second': 27.78, 'train_steps_per_second': 1.746, 'total_flos': 1640044929600000.0, 'train_loss': 0.051422107435208704, 'epoch': 10.0})

In [ ]:
# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Accuracy,Precision,Recall,F1 Label 0,F1 Label 1,F1 Label 2,F1 Label 3,F1 Label 4,F1 Label 5,F1 Label 6,F1 Label 7,F1 Label 8,F1 Label 9,F1 Label 10,F1 Label 11,F1 Label 12,F1 Label 13,F1 Label 14
0.004612,0.328844,10,0.408163,0.255818,0.281690,0.259987,0.287037,0.571429,0.250000,0.666667,0.000000,0.000000,0.000000,0.000000,0.692308,0.823529,0.000000,0.000000,0.000000,0.416667,0.416667,0.000000


{'eval_loss': 0.32884377241134644, 'eval_f1_micro': 0.40816326530612246, 'eval_f1_macro': 0.2558177117000646, 'eval_accuracy': 0.28169014084507044, 'eval_precision': 0.2599867724867725, 'eval_recall': 0.28703703703703703, 'eval_f1_label_0': 0.5714285714285714, 'eval_f1_label_1': 0.25, 'eval_f1_label_2': 0.6666666666666666, 'eval_f1_label_3': 0.0, 'eval_f1_label_4': 0.0, 'eval_f1_label_5': 0.0, 'eval_f1_label_6': 0.0, 'eval_f1_label_7': 0.6923076923076923, 'eval_f1_label_8': 0.8235294117647058, 'eval_f1_label_9': 0.0, 'eval_f1_label_10': 0.0, 'eval_f1_label_11': 0.0, 'eval_f1_label_12': 0.4166666666666667, 'eval_f1_label_13': 0.4166666666666667, 'eval_f1_label_14': 0.0}
